# View sample extractions from Qwen

Use the **extract** kernel. Run cells in order: give one paper UID, load its full Markdown, send it to Qwen with the sample-extraction prompt and compare the result with the samples already in the database. Prompts are editable below. Nothing is saved; the database and papers are only read.

## The 20 papers

These are the same 20 papers as in `synthesis_extraction.ipynb`. **The number of samples is from the previous extraction** (1–3 October 2026 run, read on 20 October 2026), not a corrected count.

| Paper UID | No. of samples | Title |
|---|---:|---|
| ID001100849 | 5 | Physics Augmented Machine Learning Discovery of Composition-Dependent Constitutive Laws for 3D Printed Digital Materials |
| ID001098910 | 7 | Polycaprolactone/graphite nanoplates composite nanopapers |
| ID001098286 | 6 | Interfacial Entropic Interactions Tunes Fragility and Dynamic Heterogeneity of Glassy Athermal Polymer Nanocomposite films |
| ID001098538 | 4 | Magnetic Shape Memory Polymers with Integrated Multifunctional Shape Manipulations |
| ID000573272 | 10 | Continuous glycolysis of waste PET via reactive extrusion and copolymerization with PTMG for enhanced toughness |
| ID000415115 | 6 | Effect of chemically modified lignin addition on the physicochemical properties of PCL nanofibers |
| ID000588793 | 11 | Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers |
| ID000581823 | 8 | Thermal properties, rheology and foams of polystyrene-block-poly(4-vinylpyridine) diblock copolymers |
| ID001002844 | 20 | Dispersant and Protective Roles of Amphiphilic Poly(ethylene phosphate) Block Copolymers in Polyester/Bone Mineral Composites. |
| ID001032337 | 6 | Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation. |
| ID000752848 | 5 | Trans crystallization behavior and strong reinforcement effect of cellulose nanocrystals on reinforced poly(butylene succinate) nanocomposites. |
| ID000180835 | 2 | A comparative investigation of uranium and thorium adsorption behavior on amidoximated copolymeric hydrogel |
| ID000180254 | 6 | Flexible Conductive Nanocomposites for Electrochemical Devices Based on Chlorinated Natural Rubber/Nickel Oxide Nanoparticles |
| ID000750264 | 2 | A surface-grafted thermoresponsive hydrogel in which the surface structure dominates the bulk properties. |
| ID000732480 | 6 | Enhancing the degradability of poly(lactic acid) through the introduction of main-chain thionoester linkages by ring-opening copolymerization |
| ID000732356 | 16 | Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester |
| ID000905310 | 24 | Green revolution in polymer composites: performance and degradability of natural fiber reinforced biopolymers |
| ID000137005 | 7 | A study of oriented conductive composites with segregated network structure obtained via solid-state processing of UHMWPE reactor powder and carbon nanofillers |
| ID000143254 | 4 | Synthesis of waterborne acrylic copolymer resin as a binding agent for the development of water-based inks in the printing application |
| ID000061162 | 4 | Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica |

In [ ]:
import sys
sys.dont_write_bytecode = True
import json
import sqlite3
import textwrap
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from either notebooks/ or the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.6-27b"
THINKING = True       # The 1–3 October run used reasoning on. Set False for thinking off.
MAX_TOKENS = 150_000  # Same cap as scripts/extract_samples_async.py.

## Prompts

Copied from the deployed sample extractor: `synth_extract/agents/extraction/prompts/sample/system_prompt.md`, `user_template.md` and `SampleExtractor.response_format()`. Edit these strings as needed; the files are not changed. Keep the `{markdown_full_text}` placeholder in `USER_PROMPT`.

In [ ]:
SYSTEM_PROMPT = """
Identify every distinct polymer material sample prepared in this paper and describe each one.

A sample is a polymer or polymer-based material made in this work. Samples are defined by how they were made: if the synthesis, formulation or processing differs (monomers, monomer ratio, composition, filler or additive loading, molecular weight, amounts, or reaction conditions such as temperature, time, catalyst, or solvent), it is a separate sample, even if the paper only reports it in a table or figure. 
Every row (entry) of a table that lists preparation conditions is a separate sample, also when the paper calls all of them by the same polymer name.
Do not merge samples that differ in synthesis. When in doubt, list it as its own sample.

A chemical modification of a sample produces a new sample. Test or measurement conditions do not create new samples.

DO NOT INCLUDE AS SAMPLES

- commercial materials used as received, only as constituents of another sample;
- individual monomers, initiators, catalysts, solvents, crosslinkers, additives, fillers, drugs, or other components when they are not themselves the sample;
- characterization instruments or measurement substrates;
- computational models;
- materials reported only from other papers;
- intermediates that are only used to prepare another sample (their synthesis belongs to that sample).

DESCRIPTION

The description should contain enough information to understand and distinguish the sample without reading the surrounding paper. Include, when reported:

1. what the material is: polymer chemistry, composition, architecture, or form;
2. how it was made: the key synthesis steps and conditions needed to identify it;
3. what distinguishes its synthesis from the other samples in the paper;
4. explicit values when they distinguish samples ("PLLA block Mn 20 kg/mol", not "longer PLLA block").

Every statement must come from the paper. Do not infer, generalize, or add background knowledge. If something is not mentioned in the paper, it does not go in the description.

FIELDS

- `name`: the polymer or material name as written in the paper (for example, "PEG-b-PLLA" or "PVDF membrane"). It is not a code or label. Different samples may share a name.
- `identifier`: the label the paper uses for this specific sample (for example, "PEL-10", "Sample 7", or "MN3"); use an empty string if none is given.
- `aliases`: every other name, code, or identifier the paper uses for this same sample.
- `description`: the self-contained description specified above.

If no qualifying samples are present, return an empty `samples` array. Return only the structured output.
"""

USER_PROMPT = """
<paper>
{markdown_full_text}
</paper>
"""

RESPONSE_FORMAT = {'type': 'json_schema',
 'json_schema': {'name': 'sample_list',
                 'strict': True,
                 'schema': {'type': 'object',
                            'properties': {'samples': {'type': 'array',
                                                       'items': {'type': 'object',
                                                                 'properties': {'name': {'type': 'string',
                                                                                         'minLength': 1},
                                                                                'identifier': {'type': 'string'},
                                                                                'aliases': {'type': 'array',
                                                                                            'items': {'type': 'string',
                                                                                                      'minLength': 1}},
                                                                                'description': {'type': 'string',
                                                                                                'minLength': 1}},
                                                                 'required': ['name',
                                                                              'identifier',
                                                                              'aliases',
                                                                              'description'],
                                                                 'additionalProperties': False}}},
                            'required': ['samples'],
                            'additionalProperties': False}}}

## Samples already in the database

Change `PAPER_UID` and run this cell. These are the records from the previous extraction, for comparison.

In [56]:
def get_samples(paper_uid):
    connection = sqlite3.connect(
        f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True
    )
    try:
        return pd.read_sql_query(
            "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
            "s.name, s.identifier, s.aliases, s.description "
            "FROM samples s JOIN papers p USING (paper_uid) "
            "WHERE s.paper_uid = ? ORDER BY s.sample_id",
            connection, params=(paper_uid,),
        )
    finally:
        connection.close()

PAPER_UID = "ID000061162"
db_samples = get_samples(PAPER_UID)
print(f"{PAPER_UID}: {len(db_samples)} samples in the database")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(db_samples)

ID000061162: 4 samples in the database


,paper_uid,source,sample_id,name,identifier,aliases,description
0,ID000061162,wiley,830062,Neat PLA,PLA,"[""neat PLA""]","Poly(lactic acid) (PLA, grade Ingeo 2003D) dried overnight at 60 °C and compression molded at 1700 psi for 3 min at 180 °C without any filler."
1,ID000061162,wiley,830063,PLA/0.77 wt% SiCO composite,0.77 wt % SiCO,"[""0.77 wt % SiCO-modified PLA composite""]","PLA composite containing 0.77 wt% capsicum oleoresin-impregnated nanoporous silica (SiCO). Prepared by mixing dried PLA with 0.77 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 0.5 wt% capsicum oleoresin (CO) relative to the total film weight."
2,ID000061162,wiley,830064,PLA/1.54 wt% SiCO composite,1.54 wt % SiCO,"[""1.54 wt % SiCO-modified PLA composite""]","PLA composite containing 1.54 wt% SiCO. Prepared by mixing dried PLA with 1.54 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 1 wt% CO relative to the total film weight."
3,ID000061162,wiley,830065,PLA/3.08 wt% SiCO composite,3.08 wt % SiCO,"[""3.08 wt % SiCO-modified PLA composite""]","PLA composite containing 3.08 wt% SiCO. Prepared by mixing dried PLA with 3.08 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 2 wt% CO relative to the total film weight."


## Choose a paper and load the full text

Set the paper UID here. This cell reads the paper's full local Markdown and builds `messages`. It does not call Qwen. You can inspect `paper_text` or `messages` before sending.

In [57]:
PAPER_UID = "ID000061162"

connection = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)
try:
    found = connection.execute(
        "SELECT canonical_source, title FROM papers WHERE paper_uid = ?", (PAPER_UID,)
    ).fetchone()
finally:
    connection.close()
if found is None:
    raise ValueError(f"Paper {PAPER_UID} is not in the database.")
source, title = found

md_path = ROOT / "data/fulltext" / source / PAPER_UID / f"{PAPER_UID}.md"
paper_text = md_path.read_text(encoding="utf-8")
if not paper_text.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")

user_message = USER_PROMPT.strip().replace("{markdown_full_text}", paper_text)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": user_message},
]
print(f"{PAPER_UID} ({source}): {title}")
print(f"Loaded full Markdown: {len(paper_text):,} characters")

ID000061162 (wiley): Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica
Loaded full Markdown: 43,459 characters


## Send to Qwen and display the samples

Each execution makes one model request. The full response remains available as `response`; the samples are in `new_samples` (a DataFrame). Nothing is written to disk.

In [58]:
async with AsyncOpenAI(
    base_url=BASE_URL, api_key="none", timeout=1800, max_retries=0
) as client:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=RESPONSE_FORMAT,
    )

answer = response.choices[0].message.content or ""
new_samples = None
try:
    parsed = json.loads(answer)
    if isinstance(parsed, dict) and isinstance(parsed.get("samples"), list):
        new_samples = pd.DataFrame(
            parsed["samples"], columns=["name", "identifier", "aliases", "description"]
        )
except json.JSONDecodeError:
    pass

if new_samples is None:
    print(answer or "[No answer content returned]")
else:
    print(f"{PAPER_UID}: {len(new_samples)} samples extracted now")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(new_samples)
if response.choices[0].finish_reason != "stop":
    print("\nResponse may be incomplete. Finish reason:", response.choices[0].finish_reason)
usage = getattr(response, "usage", None)
if usage is not None:
    print(f"\nTokens: prompt {usage.prompt_tokens:,}, completion {usage.completion_tokens:,}")

ID000061162: 4 samples extracted now


,name,identifier,aliases,description
0,PLA,PLA,[neat PLA],Poly(lactic acid) (PLA) film (grade Ingeo 2003D). The PLA was dried overnight at 60 °C and then compression molded at 1700 psi for 3 min at 180 °C. This sample contains no filler or additive.
1,PLA/SiCO composite film,0.77 wt % SiCO,[0.77 wt % SiCO-modified PLA],"PLA composite film containing 0.77 wt % capsicum oleoresin-impregnated nanoporous silica (SiCO), corresponding to 0.5 wt % capsicum oleoresin (CO) in the final film. SiCO was prepared by mixing silica (35 wt %) with acetone at 50 °C for 10 min, adding extracted CO (65 wt %), stirring for 10 min at 50 °C, and drying at 60 °C for 2 h. The dried PLA was mixed with 0.77 wt % SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, then compression molded at 1700 psi for 3 min at 180 °C."
2,PLA/SiCO composite film,1.54 wt % SiCO,[1.54 wt % SiCO-modified PLA],"PLA composite film containing 1.54 wt % SiCO, corresponding to 1 wt % CO in the final film. Prepared identically to the 0.77 wt % SiCO sample but with 1.54 wt % SiCO loading during the internal mixing step at 170 °C for 10 min at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C."
3,PLA/SiCO composite film,3.08 wt % SiCO,[3.08 wt % SiCO-modified PLA],"PLA composite film containing 3.08 wt % SiCO, corresponding to 2 wt % CO in the final film. Prepared identically to the other composites but with 3.08 wt % SiCO loading during the internal mixing step at 170 °C for 10 min at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C."



Tokens: prompt 14,252, completion 5,424


## Compare with the database

Lists the identifiers found now and in the previous extraction side by side. Run after both cells above for the same paper.

In [59]:
old = get_samples(PAPER_UID)
old_ids = [i or n for i, n in zip(old["identifier"], old["name"])]
new_ids = [] if new_samples is None else [
    i or n for i, n in zip(new_samples["identifier"], new_samples["name"])
]
rows = max(len(old_ids), len(new_ids))
comparison = pd.DataFrame({
    "previous extraction": old_ids + [""] * (rows - len(old_ids)),
    "now": new_ids + [""] * (rows - len(new_ids)),
})
print(f"previous: {len(old_ids)} samples, now: {len(new_ids)} samples")
with pd.option_context("display.max_rows", None):
    display(comparison)

previous: 4 samples, now: 4 samples


,previous extraction,now
0,PLA,PLA
1,0.77 wt % SiCO,0.77 wt % SiCO
2,1.54 wt % SiCO,1.54 wt % SiCO
3,3.08 wt % SiCO,3.08 wt % SiCO


1. for ID000732356, we see that earlier prompts squashed the fields and now we fixed them `Every row (entry) of a table that lists preparation conditions is a separate sample, also when the paper calls all of them by the same polymer name.`

## Reset the sample tables (fresh start)

**This cell writes to `data/polymer_samples.db`.** It deletes every row in `samples` and sets every paper in `sample_extraction_progress` back to `pending` (attempt count 0, errors and timestamps cleared), so the next extraction run extracts and stores all papers again. `papers` and `property_classification` are not changed.

- `CONFIRM_RESET` must be set to `True`, otherwise the cell stops without doing anything.
- `BACKUP_FIRST` writes a full copy of the database (`data/polymer_samples_backup_<date>_<time>.db`, about 1.3 GB) before deleting.
- `RESET_SAMPLE_ID_COUNTER` is off by default, so new samples get IDs after the old ones and no old `sample_id` in the synthesis runs or check sets can point to a different new sample.
- Stop all extraction jobs first; the cell refuses to run while any paper is `in_progress`.
- Deleted rows free space inside the file but do not shrink it; run `VACUUM` separately if needed.

In [ ]:
# Fresh start: delete every row in `samples` and set every paper in
# `sample_extraction_progress` back to 'pending', so the extraction jobs
# (scripts/extract_samples_async.py) extract and store all papers again.
# This WRITES to data/polymer_samples.db. The `papers` and
# `property_classification` tables are not touched.
import sqlite3
from contextlib import closing
from datetime import datetime

CONFIRM_RESET = False           # set to True to run the reset
BACKUP_FIRST = True             # copy the database before deleting anything
RESET_SAMPLE_ID_COUNTER = True # True: new samples start again at sample_id 1 (SID000001).
                                # Keep False if any file still refers to old sample_ids,
                                # otherwise an old ID could point to a different new sample.

samples_db_path = ROOT / "data" / "polymer_samples.db"
if not samples_db_path.is_file():
    raise FileNotFoundError(f"Database not found: {samples_db_path}")
if not CONFIRM_RESET:
    raise RuntimeError("Nothing done. Set CONFIRM_RESET = True to delete all samples and reset progress.")

def table_counts(conn):
    return {
        "samples": conn.execute("SELECT COUNT(*) FROM samples").fetchone()[0],
        "progress": dict(conn.execute(
            "SELECT status, COUNT(*) FROM sample_extraction_progress GROUP BY status"
        ).fetchall()),
        "next_sample_id": (conn.execute(
            "SELECT seq FROM sqlite_sequence WHERE name = 'samples'"
        ).fetchone() or (0,))[0] + 1,
    }

with closing(sqlite3.connect(samples_db_path, timeout=60)) as conn:
    before = table_counts(conn)
    if before["progress"].get("in_progress", 0):
        raise RuntimeError(
            f"{before['progress']['in_progress']} papers are 'in_progress'. "
            "Stop all extraction jobs before resetting."
        )

    if BACKUP_FIRST:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = samples_db_path.with_name(f"polymer_samples_backup_{stamp}.db")
        if backup_path.exists():
            raise FileExistsError(f"Backup already exists: {backup_path}")
        with closing(sqlite3.connect(backup_path)) as backup:
            conn.backup(backup)
        print(f"Backup written: {backup_path}")

    conn.isolation_level = None  # manage the transaction explicitly
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("BEGIN IMMEDIATE")
    try:
        conn.execute("DELETE FROM samples")
        if RESET_SAMPLE_ID_COUNTER:
            conn.execute("DELETE FROM sqlite_sequence WHERE name = 'samples'")
        # Every paper gets a progress row, then every row goes back to 'pending'.
        conn.execute(
            "INSERT OR IGNORE INTO sample_extraction_progress (paper_uid) "
            "SELECT paper_uid FROM papers"
        )
        conn.execute(
            """
            UPDATE sample_extraction_progress
            SET status = 'pending',
                attempt_count = 0,
                last_error = NULL,
                started_at = NULL,
                completed_at = NULL,
                updated_at = CURRENT_TIMESTAMP
            """
        )
        conn.execute("COMMIT")
    except Exception:
        conn.execute("ROLLBACK")
        raise

    after = table_counts(conn)
    paper_count = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]

print(f"Database: {samples_db_path}")
print(f"samples rows: {before['samples']:,} -> {after['samples']:,}")
print(f"progress before: {before['progress']}")
print(f"progress after:  {after['progress']} / papers: {paper_count:,}")
print(f"next sample_id: {after['next_sample_id']:,}")

Backup written: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples_backup_20261026_143352.db
Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
samples rows: 1,816,337 -> 0
progress before: {'failed': 4, 'success': 273187}
progress after:  {'pending': 273191} / papers: 273,191
next sample_id: 1


## Extraction progress

Read-only summary of the run in `data/polymer_samples.db`: status counts, attempts, samples per paper, failures grouped by error type, papers currently `in_progress`, consistency checks and claim-to-result times. Safe to run while jobs are writing.

In [71]:
# Progress of the sample-extraction run in data/polymer_samples.db (read-only).
import sqlite3
from contextlib import closing

import pandas as pd
from IPython.display import display

stats_db_path = ROOT / "data" / "polymer_samples.db"
stats_uri = f"{stats_db_path.as_uri()}?mode=ro"

with closing(sqlite3.connect(stats_uri, uri=True, timeout=60)) as con:
    con.execute("PRAGMA query_only = ON")

    def q(sql, params=()):
        return pd.read_sql_query(sql, con, params=params)

    n_papers = q("SELECT COUNT(*) AS n FROM papers").n[0]
    n_progress = q("SELECT COUNT(*) AS n FROM sample_extraction_progress").n[0]
    n_samples = q("SELECT COUNT(*) AS n FROM samples").n[0]

    print(f"Database: {stats_db_path}")
    print(f"papers={n_papers:,}  progress rows={n_progress:,}  sample rows={n_samples:,}\n")

    # 1. Progress status
    status_df = q(
        """
        SELECT status, COUNT(*) AS papers,
               ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM sample_extraction_progress), 2) AS pct
        FROM sample_extraction_progress GROUP BY status ORDER BY papers DESC
        """
    )
    print("== Extraction status ==")
    display(status_df)

    attempts_df = q(
        "SELECT attempt_count, status, COUNT(*) AS papers FROM sample_extraction_progress "
        "WHERE attempt_count > 0 GROUP BY attempt_count, status ORDER BY attempt_count, status"
    )
    print("== Attempts per paper (attempted papers only) ==")
    display(attempts_df)

    # 2. Samples extracted from successful papers
    per_paper = q(
        """
        SELECT p.paper_uid, COUNT(s.sample_id) AS n_samples
        FROM sample_extraction_progress p
        LEFT JOIN samples s USING (paper_uid)
        WHERE p.status = 'success'
        GROUP BY p.paper_uid
        """
    )
    n_success = len(per_paper)
    print("== Samples from successful papers ==")
    print(f"successful papers:              {n_success:,}")
    print(f"total samples extracted:        {int(per_paper.n_samples.sum()):,}")
    print(f"papers with >=1 sample:         {(per_paper.n_samples > 0).sum():,}")
    print(f"papers with 0 samples (success):{(per_paper.n_samples == 0).sum():,}")
    if n_success:
        print(f"mean / median / max per paper:  {per_paper.n_samples.mean():.2f} / "
              f"{per_paper.n_samples.median():.0f} / {per_paper.n_samples.max()}")
        labels = ["0", "1", "2", "3", "4-5", "6-10", "11-20", "21-50", "51-100", ">100"]
        bins = pd.cut(
            per_paper.n_samples,
            bins=[-1, 0, 1, 2, 3, 5, 10, 20, 50, 100, 10**9],
            labels=labels,
        )
        dist = bins.value_counts().reindex(labels).fillna(0).astype(int)
        print("\nSamples-per-paper distribution:")
        display(dist.rename("papers").to_frame())
        print("Papers with the most samples:")
        display(per_paper.sort_values("n_samples", ascending=False).head(10))

    # 3. Failures
    failed_df = q(
        "SELECT paper_uid, attempt_count, last_error FROM sample_extraction_progress "
        "WHERE status = 'failed'"
    )
    print(f"== Failures: {len(failed_df):,} papers ==")
    if len(failed_df):
        failed_df["error_type"] = (
            failed_df.last_error.fillna("(none)").str.split(":", n=1).str[0].str.slice(0, 60)
        )
        display(failed_df.error_type.value_counts().rename("papers").to_frame())
        for etype, group in failed_df.groupby("error_type"):
            example = group.iloc[0]
            print(f"- {etype}: e.g. {example.paper_uid} -> {str(example.last_error)[:300]}")
    stuck_df = q(
        "SELECT paper_uid, attempt_count, started_at FROM sample_extraction_progress "
        "WHERE status = 'in_progress' ORDER BY started_at"
    )
    print(f"\n== in_progress right now: {len(stuck_df):,} papers ==")
    if len(stuck_df):
        display(stuck_df.head(10))

    # 4. Consistency checks
    checks = {
        "progress rows vs papers (should differ by 0)": int(n_progress - n_papers),
        "sample rows whose paper has no success status": int(q(
            "SELECT COUNT(*) AS n FROM samples s JOIN sample_extraction_progress p USING (paper_uid) "
            "WHERE p.status != 'success'").n[0]),
        "papers with samples but status != success": int(q(
            "SELECT COUNT(DISTINCT s.paper_uid) AS n FROM samples s "
            "JOIN sample_extraction_progress p USING (paper_uid) WHERE p.status != 'success'").n[0]),
        "sample rows with unknown paper_uid": int(q(
            "SELECT COUNT(*) AS n FROM samples s LEFT JOIN papers p USING (paper_uid) "
            "WHERE p.paper_uid IS NULL").n[0]),
        "success rows without completed_at": int(q(
            "SELECT COUNT(*) AS n FROM sample_extraction_progress "
            "WHERE status = 'success' AND completed_at IS NULL").n[0]),
        "pending rows with attempts but never completed (released claims)": int(q(
            "SELECT COUNT(*) AS n FROM sample_extraction_progress "
            "WHERE status = 'pending' AND attempt_count > 0").n[0]),
        "empty name or description in samples": int(q(
            "SELECT COUNT(*) AS n FROM samples WHERE TRIM(name) = '' OR TRIM(description) = ''").n[0]),
    }
    print("\n== Consistency checks (0 means OK) ==")
    for label, value in checks.items():
        print(f"{'OK ' if value == 0 else 'CHECK'}  {label}: {value:,}")

    sum_per_paper = int(per_paper.n_samples.sum()) if n_success else 0
    print(f"\nsamples table rows ({n_samples:,}) == sum of samples over success papers "
          f"({sum_per_paper:,}): {n_samples == sum_per_paper}")

    # 5. Throughput
    timing = q(
        """
        SELECT (julianday(completed_at) - julianday(started_at)) * 86400 AS seconds
        FROM sample_extraction_progress
        WHERE status IN ('success', 'failed') AND completed_at IS NOT NULL AND started_at IS NOT NULL
        """
    )
    if len(timing):
        print("\n== Seconds from claim to result (includes batch wait) ==")
        display(timing.seconds.describe().round(1).to_frame("seconds"))

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
papers=273,191  progress rows=273,191  sample rows=1,954,131

== Extraction status ==


,status,papers,pct
0,success,273190,100.0
1,failed,1,0.0


== Attempts per paper (attempted papers only) ==


,attempt_count,status,papers
0,1,success,273083
1,2,success,88
2,3,success,9
3,4,success,6
4,5,failed,1
5,5,success,4


== Samples from successful papers ==
successful papers:              273,190
total samples extracted:        1,954,131
papers with >=1 sample:         271,522
papers with 0 samples (success):1,668
mean / median / max per paper:  7.15 / 5 / 496

Samples-per-paper distribution:


,papers
n_samples,
0,1668
1,21550
2,28828
3,28706
4-5,62819
6-10,79120
11-20,38641
21-50,11160
51-100,628


... (2 outputs left out)

== Failures: 1 papers ==


,papers
error_type,
invalid_response,1


- invalid_response: e.g. ID000167793 -> invalid_response: The response did not match task 'sample_extraction': ValidationError: 1 validation error for SampleList
  Invalid JSON: control character (\u0000-\u001F) found while parsing a string at line 30 column 26 [type=json_invalid, input_value='{\n  "samples": [\n    {...inutes."\n    }\n 

== in_progress right now: 0 papers ==

== Consistency checks (0 means OK) ==
OK   progress rows vs papers (should differ by 0): 0
OK   sample rows whose paper has no success status: 0
OK   papers with samples but status != success: 0
OK   sample rows with unknown paper_uid: 0
OK   success rows without completed_at: 0
OK   pending rows with attempts but never completed (released claims): 0
OK   empty name or description in samples: 0

samples table rows (1,954,131) == sum of samples over success papers (1,954,131): True

== Seconds from claim to result (includes batch wait) ==


,seconds
count,273191.0
mean,525.1
std,171.6
min,61.0
25%,464.0
50%,516.0
75%,565.0
max,3920.0


## Progress of the 20 pilot papers

Read-only. For each pilot paper: status, samples extracted now, samples in the previous extraction (`data/polymer_samples_26_10.db`, if present), its manifest file and line (processing order within that job), attempts, timestamps and the last error.

In [67]:
# Progress of the 20 pilot papers (read-only). Shows each paper's status, the
# number of samples extracted now, where it sits in the UID manifests, and the
# number of samples in the previous extraction (backup database), if present.
import sqlite3
from contextlib import closing

import pandas as pd
from IPython.display import display

PILOT_UIDS = [
    "ID000061162", "ID000137005", "ID000143254", "ID000180254", "ID000180835",
    "ID000415115", "ID000573272", "ID000581823", "ID000588793", "ID000732356",
    "ID000732480", "ID000750264", "ID000752848", "ID000905310", "ID001002844",
    "ID001032337", "ID001098286", "ID001098538", "ID001098910", "ID001100849",
]
pilot_db_path = ROOT / "data" / "polymer_samples.db"
previous_db_path = ROOT / "data" / "polymer_samples_26_10.db"  # old extraction, before the reset
uid_manifest_dir = ROOT / "data" / "uid_manifest"

def read_only(path):
    return closing(sqlite3.connect(f"{path.as_uri()}?mode=ro", uri=True, timeout=60))

placeholders = ", ".join("?" for _ in PILOT_UIDS)
with read_only(pilot_db_path) as con:
    pilot = pd.read_sql_query(
        f"""
        SELECT p.paper_uid, p.status, p.attempt_count, p.started_at, p.completed_at,
               (SELECT COUNT(*) FROM samples s WHERE s.paper_uid = p.paper_uid) AS n_samples_now,
               SUBSTR(p.last_error, 1, 120) AS last_error
        FROM sample_extraction_progress p
        WHERE p.paper_uid IN ({placeholders})
        """,
        con,
        params=PILOT_UIDS,
    )

if previous_db_path.is_file():
    with read_only(previous_db_path) as con:
        previous = pd.read_sql_query(
            f"SELECT paper_uid, COUNT(*) AS n_samples_previous FROM samples "
            f"WHERE paper_uid IN ({placeholders}) GROUP BY paper_uid",
            con,
            params=PILOT_UIDS,
        )
    pilot = pilot.merge(previous, on="paper_uid", how="left")
    pilot["n_samples_previous"] = pilot["n_samples_previous"].fillna(0).astype(int)

# Manifest file and line (= processing order within that job) for each paper.
location = {}
for manifest in sorted(uid_manifest_dir.glob("sample_extraction_uids_*.txt")):
    lines = [line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()]
    for line_number, uid in enumerate(lines, start=1):
        if uid in PILOT_UIDS:
            location[uid] = (manifest.name, f"{line_number} / {len(lines)}")
pilot["manifest"] = pilot.paper_uid.map(lambda uid: location.get(uid, ("", ""))[0])
pilot["line"] = pilot.paper_uid.map(lambda uid: location.get(uid, ("", ""))[1])

missing = sorted(set(PILOT_UIDS) - set(pilot.paper_uid))
if missing:
    print(f"Not in sample_extraction_progress: {missing}")

order = {uid: i for i, uid in enumerate(PILOT_UIDS)}
pilot = pilot.sort_values("paper_uid", key=lambda s: s.map(order)).reset_index(drop=True)
done = (pilot.status == "success").sum()
print(f"Database: {pilot_db_path}")
print(f"Pilot papers extracted: {done} / {len(PILOT_UIDS)}   "
      f"status counts: {pilot.status.value_counts().to_dict()}")
columns = ["paper_uid", "status", "n_samples_now"]
if "n_samples_previous" in pilot:
    columns.append("n_samples_previous")
columns += ["manifest", "line", "attempt_count", "started_at", "completed_at", "last_error"]
display(pilot[columns])

Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
Pilot papers extracted: 20 / 20   status counts: {'success': 20}


,paper_uid,status,n_samples_now,n_samples_previous,manifest,line,attempt_count,started_at,completed_at,last_error
0,ID000061162,success,4,4,sample_extraction_uids_0009.txt,1984 / 3000,1,2026-10-27 02:38:34,2026-10-27 02:47:51,None
1,ID000137005,success,8,7,sample_extraction_uids_0017.txt,1513 / 3000,1,2026-10-26 23:50:30,2026-10-26 23:59:41,None
2,ID000143254,success,4,4,sample_extraction_uids_0018.txt,2508 / 3000,1,2026-10-27 06:28:12,2026-10-27 06:37:58,None
3,ID000180254,success,6,6,sample_extraction_uids_0022.txt,2916 / 3000,1,2026-10-27 06:53:19,2026-10-27 07:00:28,None
4,ID000180835,success,2,2,sample_extraction_uids_0023.txt,264 / 3000,1,2026-10-26 15:54:03,2026-10-26 16:01:13,None
5,ID000415115,success,6,6,sample_extraction_uids_0042.txt,806 / 3000,1,2026-10-26 19:13:57,2026-10-26 19:22:15,None
6,ID000573272,success,5,10,sample_extraction_uids_0057.txt,745 / 3000,1,2026-10-26 18:55:19,2026-10-26 19:03:14,None
7,ID000581823,success,8,8,sample_extraction_uids_0059.txt,1245 / 3000,1,2026-10-26 22:41:19,2026-10-26 22:51:39,None
8,ID000588793,success,13,11,sample_extraction_uids_0061.txt,545 / 3000,1,2026-10-26 17:53:27,2026-10-26 18:04:34,None
9,ID000732356,success,29,16,sample_extraction_uids_0072.txt,2801 / 3000,1,2026-10-27 08:59:40,2026-10-27 09:08:37,None
